# PVGIS CATCH — training 2005–2018, target 2019

Notebook riproducibile per eseguire il detector multivariato CATCH sui dati PVGIS.

- training: anni completi **2005–2018**;
- test strettamente held-out: **2019**;
- un modello indipendente per località;
- soglia calibrata esclusivamente sugli score di training;
- finestre separate per anno e copertura completa della coda.

La logica resta in `physiq_pv.experiments.pvgis_catch_pipeline`: il notebook configura,
esegue e visualizza gli output senza duplicare l'implementazione. L'esecuzione completa
può richiedere molto tempo; per una verifica rapida impostare `MAX_LOCATIONS = 1`.

## 1. Setup

In [ ]:
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from physiq_pv.experiments.pvgis_catch_pipeline import (
    PVGISCATCHConfig,
    run_pvgis_catch,
)

print("Repository:", REPO_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

## 2. Configurazione

Modificare `PVGIS_DIR` se i NetCDF si trovano altrove. È possibile impostare anche
la variabile d'ambiente `PVGIS_DIR`. Lasciare `MAX_LOCATIONS = None` per il run completo.

In [ ]:
PVGIS_DIR = Path(os.environ.get(
    "PVGIS_DIR",
    "/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance",
)).expanduser()
OUT_DIR = REPO_ROOT / "outputs" / "pvgis_catch_2005_2019"
FILE_TEMPLATE = "piedmont_pvgis_{year}.nc"

TRAIN_YEARS = tuple(range(2005, 2019))
EXPORT_TRAIN_YEARS = (2016, 2017, 2018)  # etichette per SDE normal-only
TEST_YEAR = 2019
MAX_LOCATIONS = None  # usare 1 per uno smoke test rapido
RUN_EXPERIMENT = True
ALLOW_OVERWRITE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = PVGISCATCHConfig(
    pvgis_dir=str(PVGIS_DIR),
    train_years=TRAIN_YEARS,
    export_train_years=EXPORT_TRAIN_YEARS,
    test_year=TEST_YEAR,
    file_template=FILE_TEMPLATE,
    out_dir=str(OUT_DIR),
    max_locations=MAX_LOCATIONS,
    device=DEVICE,
    # Parametri CATCH paper-first / repository per dettagli non specificati:
    seq_len=192,
    patch_size=16,
    patch_stride=8,
    inference_patch_size=32,
    inference_patch_stride=1,
    mask_source="projected",
    epochs=3,
    batch_size=32,
    learning_rate=1e-4,
    mask_learning_rate=1e-5,
    contamination=0.01,
    seed=42,
)

display(pd.Series(asdict(config), name="valore").to_frame())

## 3. Controllo dei dati

Il protocollo richiede tutti i file annuali dal 2005 al 2019. Il training si interrompe
se manca anche un solo anno, evitando un intervallo storico incompleto.

In [ ]:
required_years = (*TRAIN_YEARS, TEST_YEAR)
year_paths = {
    year: PVGIS_DIR / FILE_TEMPLATE.format(year=year)
    for year in required_years
}
availability = pd.DataFrame({
    "year": required_years,
    "split": ["train" if year in TRAIN_YEARS else "test" for year in required_years],
    "path": [str(year_paths[year]) for year in required_years],
    "exists": [year_paths[year].is_file() for year in required_years],
})
display(availability)

missing = availability.loc[~availability["exists"], "path"].tolist()
if missing:
    raise FileNotFoundError(
        "File PVGIS mancanti. Correggere PVGIS_DIR o completare i dati:\n- "
        + "\n- ".join(missing)
    )

assert TRAIN_YEARS == tuple(range(2005, 2019))
assert TEST_YEAR == 2019 and TEST_YEAR not in TRAIN_YEARS
print("OK: 14 anni di training e anno target 2019 presenti.")

## 4. Esecuzione

`RUN_EXPERIMENT = True` avvia il fit completo. La pipeline addestra un modello CATCH
separato per ogni località e salva progressivamente gli output finali nella directory scelta.

In [ ]:
if RUN_EXPERIMENT:
    if OUT_DIR.exists() and any(OUT_DIR.iterdir()) and not ALLOW_OVERWRITE:
        raise FileExistsError(
            f"Output non vuoto: {OUT_DIR}. Scegliere una nuova directory o impostare "
            "ALLOW_OVERWRITE=True esplicitamente."
        )
    paths = run_pvgis_catch(config)
    print("\nOutput prodotti:")
    for name, path in paths.items():
        print(f"- {name}: {path}")
else:
    print("Esecuzione disattivata. Impostare RUN_EXPERIMENT = True e rieseguire.")

## 5. Caricamento e verifica degli output

In [ ]:
output_paths = {
    "scores": OUT_DIR / "catch_scores.csv",
    "anomaly_scores": OUT_DIR / "anomaly_scores.csv",
    "train_anomaly_scores": OUT_DIR / "train_anomaly_scores.csv",
    "detections": OUT_DIR / "catch_detections.csv",
    "intervals": OUT_DIR / "catch_intervals.csv",
    "summary": OUT_DIR / "catch_location_summary.csv",
    "meta": OUT_DIR / "catch_meta.json",
    "report": OUT_DIR / "report.md",
}
missing_outputs = [str(path) for path in output_paths.values() if not path.is_file()]
if missing_outputs:
    raise FileNotFoundError(
        "Output mancanti: eseguire prima la sezione 4.\n- "
        + "\n- ".join(missing_outputs)
    )

scores = pd.read_csv(output_paths["scores"], parse_dates=["timestamp"])
detections = pd.read_csv(output_paths["detections"], parse_dates=["timestamp"])
intervals = pd.read_csv(output_paths["intervals"], parse_dates=["start", "end"])
summary = pd.read_csv(output_paths["summary"])
meta = json.loads(output_paths["meta"].read_text(encoding="utf-8"))

assert meta["train_years"] == list(TRAIN_YEARS)
assert meta["export_train_years"] == list(EXPORT_TRAIN_YEARS)
assert meta["test_year"] == TEST_YEAR
assert meta["threshold_calibration"] == "training-only quantile"
assert set(scores["timestamp"].dt.year.unique()) == {TEST_YEAR}
assert scores["global_score"].notna().all()
assert scores.groupby("location")["threshold"].nunique().le(1).all()

print("Righe score:", len(scores))
print("Località:", scores["location"].nunique())
print("Anomalie:", int(scores["is_anomaly"].sum()))
display(summary)

## 6. Score e anomalie per località

In [ ]:
SELECTED_LOCATION = str(summary.iloc[0]["location"])
location_scores = scores[
    scores["location"].astype(str) == SELECTED_LOCATION
].sort_values("timestamp")
location_anomalies = location_scores[location_scores["is_anomaly"].astype(bool)]

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(
    location_scores["timestamp"],
    location_scores["global_score"],
    linewidth=0.8,
    label="CATCH global score",
)
ax.axhline(
    location_scores["threshold"].iloc[0],
    color="tab:orange",
    linestyle="--",
    label="soglia train-only",
)
ax.scatter(
    location_anomalies["timestamp"],
    location_anomalies["global_score"],
    color="tab:red",
    s=10,
    label="anomalia",
    zorder=3,
)
ax.set(title=f"CATCH — località {SELECTED_LOCATION}, anno {TEST_YEAR}", ylabel="score")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
top_columns = [
    "location",
    "timestamp",
    "global_score",
    "time_score",
    "frequency_score",
    "threshold",
    "top_sensor",
]
display(
    detections.sort_values("global_score", ascending=False)
    .loc[:, top_columns]
    .head(20)
)

print("Intervalli anomali più intensi:")
display(intervals.sort_values("max_global_score", ascending=False).head(20))

## 7. Report riproducibile

In [ ]:
display(Markdown(output_paths["report"].read_text(encoding="utf-8")))
print("Metadati completi:")
display(pd.Series(meta, name="valore").to_frame())